In [ ]:
import time
import pandas as pd
import numpy as np
import math
from pybit.unified_trading import HTTP
from datetime import datetime
import sys

# ==========================================
# 🏆 TITAN PAIRS: RISK MANAGER EDITION
# ==========================================
print("--- 🏛️ TITAN STATISTICAL ARBITRAGE BOT ---")
print("    Account: Bybit Unified Demo")
print("    Risk System: ACTIVE (Circuit Breaker Enabled)")

# 1. YOUR KEYS
API_KEY = "L5tRvVOKUf28F1Lags"
API_SECRET = "LfUDzriifjcur6uTJA4eDxc3VXFb2kWTwG07"

# 2. CONFIGURATION
COIN_A = "SOLUSDT"
COIN_B = "LINKUSDT"
TIMEFRAME = "60"      
WINDOW = 40           
Z_ENTRY = 2.0         # Enter Trade here
Z_EXIT = 0.2          # 🎯 TARGET: Take Profit when gap closes to 0.2
Z_STOP = 4.0          # 🛡️ STOP LOSS: Close if gap widens to 4.0
LEVERAGE = 10         
PCT_SIZE = 0.05       # 5% Position Size

# ==========================================
# ⏳ THE TIME WARPER & URL FIX
# ==========================================
class TitanSession(HTTP):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.endpoint = "https://api-demo.bybit.com"

print("\n📡 SYNCING CLOCKS...")
try:
    temp_session = TitanSession(testnet=True, api_key=API_KEY, api_secret=API_SECRET)
    server_time = int(temp_session.get_server_time()['result']['timeSecond'])
    local_time = int(time.time())
    offset = server_time - local_time
    
    _real_time = time.time
    def _fake_time(): return _real_time() + offset
    time.time = _fake_time
    print(f"✅ TIME SYNCED. Drift: {offset}s")

except Exception as e:
    print(f"⚠️ Sync Warning: {e}. Proceeding with fallback.")

# ==========================================
# 🔌 CONNECT
# ==========================================
print("\n📡 INITIALIZING SESSION...")
session = None
try:
    session = TitanSession(
        testnet=True,
        api_key=API_KEY,
        api_secret=API_SECRET,
        recv_window=20000
    )
    wallet = session.get_wallet_balance(accountType="UNIFIED", coin="USDT")
    equity = float(wallet['result']['list'][0]['totalEquity'])
    print(f"✅ CONNECTED. Equity: ${equity:.2f}")

except Exception as e:
    print(f"\n❌ CONNECTION FAILED: {e}")
    sys.exit()

# Set Leverage
print("   Setting Leverage...", end=" ")
for symbol in [COIN_A, COIN_B]:
    try:
        session.set_leverage(category="linear", symbol=symbol, buyLeverage=str(LEVERAGE), sellLeverage=str(LEVERAGE))
    except: pass 
print("Done.")

# ==========================================
# 🧠 MATH ENGINE
# ==========================================
def get_z_score():
    try:
        limit = WINDOW + 10
        kline_a = session.get_kline(category="linear", symbol=COIN_A, interval=TIMEFRAME, limit=limit)
        df_a = pd.DataFrame(kline_a['result']['list'], columns=['t','o','h','l','c','v','turn'])
        df_a = df_a.iloc[::-1].reset_index(drop=True).astype(float)
        
        kline_b = session.get_kline(category="linear", symbol=COIN_B, interval=TIMEFRAME, limit=limit)
        df_b = pd.DataFrame(kline_b['result']['list'], columns=['t','o','h','l','c','v','turn'])
        df_b = df_b.iloc[::-1].reset_index(drop=True).astype(float)
        
        min_len = min(len(df_a), len(df_b))
        ratio = df_a['c'].iloc[-min_len:] / df_b['c'].iloc[-min_len:]
        
        rolling_mean = ratio.rolling(window=WINDOW).mean()
        rolling_std = ratio.rolling(window=WINDOW).std()
        z_score = (ratio - rolling_mean) / rolling_std
        
        return z_score.iloc[-1], ratio.iloc[-1], df_a['c'].iloc[-1], df_b['c'].iloc[-1]
    except: return 0, 0, 0, 0

def place_pairs_trade(side_a, side_b, price_a, price_b):
    try:
        w = session.get_wallet_balance(accountType="UNIFIED", coin="USDT")
        eq = float(w['result']['list'][0]['totalEquity'])
        trade_val_usdt = (eq * PCT_SIZE) / 2
        
        # Force Integers for Precision
        qty_a = int(math.floor((trade_val_usdt * LEVERAGE) / price_a))
        qty_b = int(math.floor((trade_val_usdt * LEVERAGE) / price_b))
        
        print(f"\n🚀 SIGNAL FIRED: {side_a} {qty_a} SOL / {side_b} {qty_b} LINK")
        
        session.place_order(category="linear", symbol=COIN_A, side=side_a, orderType="Market", qty=str(qty_a))
        session.place_order(category="linear", symbol=COIN_B, side=side_b, orderType="Market", qty=str(qty_b))
        print("✅ ORDERS SENT.")

    except Exception as e: print(f"❌ Execution Error: {e}")

def close_all_positions(reason="TARGET"):
    try:
        print(f"\n🛑 CLOSING ALL POSITIONS ({reason})...")
        for coin in [COIN_A, COIN_B]:
            pos = session.get_positions(category="linear", symbol=coin)['result']['list'][0]
            if float(pos['size']) > 0:
                side = "Sell" if pos['side'] == "Buy" else "Buy"
                session.place_order(category="linear", symbol=coin, side=side, orderType="Market", qty=pos['size'])
        print("✅ CLOSED.")
    except: pass

# ==========================================
# 🏁 MAIN LOOP
# ==========================================
print(f"📡 MONITORING MARKET...")

while True:
    try:
        z, r, price_a, price_b = get_z_score()
        now = datetime.now().strftime("%H:%M:%S")
        
        status = "NEUTRAL"
        try:
            pos_a = session.get_positions(category="linear", symbol=COIN_A)['result']['list'][0]
            if float(pos_a['size']) > 0: status = "ACTIVE TRADE"
        except: pass

        print(f"\r⏳ {now} | Z-Score: {z:.3f} | Status: {status}    ", end="")
        
        # 1. 🛡️ STOP LOSS CHECK (Circuit Breaker)
        if abs(z) > Z_STOP and status == "ACTIVE TRADE":
             print(f"\n⛔ DANGER: Z-Score hit {z:.2f}. Triggering Circuit Breaker!")
             close_all_positions(reason="STOP LOSS")
             time.sleep(10)

        # 2. ENTRY SIGNALS
        elif z > Z_ENTRY and status == "NEUTRAL":
            print("\n📈 SIGNAL: Short SOL, Long LINK.")
            place_pairs_trade("Sell", "Buy", price_a, price_b)
            time.sleep(5)
            
        elif z < -Z_ENTRY and status == "NEUTRAL":
            print("\n📉 SIGNAL: Long SOL, Short LINK.")
            place_pairs_trade("Buy", "Sell", price_a, price_b)
            time.sleep(5)
            
        # 3. 🎯 TARGET SIGNAL (Take Profit)
        elif abs(z) < Z_EXIT and status == "ACTIVE TRADE":
            print("\n🎯 TARGET REACHED. Closing.")
            close_all_positions(reason="PROFIT")
            time.sleep(5)
            
        sys.stdout.flush()
        time.sleep(10) 
        
    except KeyboardInterrupt:
        print("\n🛑 Stopped.")
        break
    except Exception as e:
        print(f"Loop Error: {e}")
        time.sleep(10)